# Preprocessing & Feature Engineering

This notebook performs all preprocessing steps required before model training.  
- Train/Test Split  
- Scaling of Time and Amount  
- SMOTE Oversampling  
- Random Forest Feature Selection  
- Applying Selected Features to Train and Test Sets  

These steps ensure the dataset is balanced, standardized, and reduced to the most informative features.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

from imblearn.over_sampling import SMOTE

data = pd.read_csv("../data/creditcard.csv")
data.head()

Dataset sets are loaded and prepared for preprocessing.  
The target variable is `Class`, where 1 = Fraud and 0 = Non‑Fraud.

In [ ]:
X = data.drop("Class", axis=1)
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

Stratified splitting is used to preserve the fraud ratio in both training and test sets.

In [ ]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

for col in ["Time", "Amount"]:
    X_train_scaled[col] = scaler.fit_transform(X_train[[col]])
    X_test_scaled[col] = scaler.transform(X_test[[col]])

X_train_scaled.head()

Time and Amount are not PCA‑transformed features, so they must be scaled.  
All other features (V1–V28) are already PCA‑transformed and do not require scaling.

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

y_train.value_counts(), y_train_res.value_counts()

The dataset is highly imbalanced.  
SMOTE oversamples the minority class (fraud) to help the model learn fraud patterns more effectively.

In [ ]:
rf_fs = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_fs.fit(X_train_res, y_train_res)

selector = SelectFromModel(rf_fs, threshold="median", prefit=True)

X_train_sel = selector.transform(X_train_res)
X_test_sel = selector.transform(X_test_scaled)

selected_features = X_train_scaled.columns[selector.get_support()]
selected_features

A Random Forest model is used to compute feature importances.

`SelectFromModel` keeps only features above the median importance score.  
This reduces dimensionality and improves model performance.

The selected features will be used in Notebook 03 for model training.

In [ ]:
pd.Series(selected_features).to_csv("../reports/selected_features.csv", index=False)